In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 6),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NYC_tz = pytz.timezone("America/New_York") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedStructure, UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

curve_mdp = IRSwapsMDP(source="GSQUANT-RL")
usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")

ts_builder = TimeseriesBuilder()

In [4]:
start = datetime.date(2010, 1, 1)
end = datetime.date(2026, 3, 27)

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[
  		UnifiedQuery(
            curve="USD-OIS",
            tenor="10Y",
            value=UnifiedValue.IRS_RATE,
        ),
        UnifiedQuery(
            cusip="CT10",
            value=UnifiedValue.FRB_YTM,
        ),       
    ],
    n_jobs=12,
	routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
		"FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
	},
    ignore_cache_miss=True,
)
df

WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb
WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb
VECTOR PRICING USD-OIS IRS...:   0%|          | 0/159 [00:00<?, ?it/s]                  

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LOADING USD-OIS raw curves...:   0%|          | 0/159 [00:00<?, ?it/s]          

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FETCHING DATA FROM WSJ...: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]
ERROR	Thread(ts-builder_1) FixedRateBondsTB:FixedRateBondsTB.py:get_timeseries()- bulk_get_data failed. Falling back per-date. Error: 'No FedInvest snapshot for 2026-03-25 (key: 2026-03-25 00:00:00)'
 Traceback (most recent call last):
   File "c:\Users\chris\clee\ARBS\notebooks\timeseries\../..\TB\FixedRateBondsTB.py", line 329, in get_timeseries
    bulk_map: Dict[DateLike, Dict[str, _FixedRateBondGenericPricer]] = self.mdp.bulk_get_data(
                                                                       ~~~~~~~~~~~~~~~~~~~~~~^
        timestamps=to_price_dates,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
        max_workers=bulk_workers,
        ^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
   File "c:\Users\chris\clee\ARBS\notebooks\timeseries\../..\MDP\FixedRateBonds\FixedRateBondsMDP.py", line 2003, in bulk_get_data
    results.append(fut.result())
                   ~~~~~~~~~~^^
   File "c:\Users\chr

,USD-OIS 10Y OUTRIGHT RATE,CT10 OUTRIGHT YTM
Date,,
2010-01-04,3.761570,3.830633
2010-01-05,3.656360,3.752607
2010-01-06,3.712310,3.826907
2010-01-07,3.738870,3.827005
2010-01-08,3.715530,3.807697
...,...,...
2026-03-23,3.809673,4.339590
2026-03-24,3.855310,4.389000
2026-03-25,3.806072,4.331000


In [6]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    df["USD-OIS 10Y OUTRIGHT RATE"],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
plot(
    df["CT10 OUTRIGHT YTM"],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
legend(show_date=True, loc="upper left")
plt.show()